# 06 多變項分析 — 參考解答

用松柏護理之家退伍軍人症 line list 練習 Modified Poisson regression（adjusted RR）
和邏輯斯迴歸（adjusted OR），並比較兩者差異。

In [ ]:
# Google Colab setup -- 若在本機執行可跳過此 cell
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
import pathlib

import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import warnings

# -- CJK font setup (避免中文標籤顯示為方框) --
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150

# --- 讀取資料 ---
df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)
fs_map = {"bedridden": 0, "assisted": 1, "independent": 2}
df["functional_score"] = df["functional_status"].map(fs_map)

## 題目 1：死亡預測 — Crude RR vs Crude OR

1. 建立 `dead` 欄位
2. 計算死亡率（case fatality rate）
3. 同時計算 crude RR（Modified Poisson）和 crude OR（logistic）
4. 整理成對照表格

In [ ]:
# --- 建立結果變項 ---
df["dead"] = (df["outcome"] == "dead").astype(int)

# 嚴重度轉數值（未感染者設為 0）
sev_map = {"not_ill": 0, "asymptomatic": 0, "mild": 1, "moderate": 2, "severe": 3}
df["severity_score"] = df["clinical_severity"].map(sev_map)

# 只用感染者做死亡預測（未感染者不會死於此疾病）
cases = df[df["infected"] == 1].copy()
cfr = cases["dead"].mean()
print(f"感染者：{len(cases)} 人，死亡：{cases['dead'].sum()} 人")
print(f"致死率 (CFR)：{cfr:.1%}")
print(f"→ CFR = {cfr:.1%}，比侵襲率 43% 低很多")
print(f"→ 預期 OR 和 RR 的差距會比感染預測時小\n")

# --- 同時計算 crude RR 和 crude OR ---
factors_death = ["age", "comorbidity_chf", "comorbidity_copd",
                 "immunosuppressed", "severity_score"]

crude_rows = []
for var in factors_death:
    # Modified Poisson → crude RR
    poisson = smf.glm(
        f"dead ~ {var}", data=cases,
        family=sm.families.Poisson()
    ).fit(cov_type="HC0", disp=0)
    rr = np.exp(poisson.params[var])
    rr_ci = np.exp(poisson.conf_int().loc[var])

    # Logistic → crude OR
    logit = smf.logit(f"dead ~ {var}", data=cases).fit(disp=0)
    or_val = np.exp(logit.params[var])
    or_ci = np.exp(logit.conf_int().loc[var])

    crude_rows.append({
        "variable": var,
        "crude_RR": round(rr, 3),
        "RR 95% CI": f"{rr_ci[0]:.3f}\u2013{rr_ci[1]:.3f}",
        "crude_OR": round(or_val, 3),
        "OR 95% CI": f"{or_ci[0]:.3f}\u2013{or_ci[1]:.3f}",
    })

crude_df = pd.DataFrame(crude_rows)
print("=== 死亡預測：Crude RR vs Crude OR ===")
print(crude_df.to_string(index=False))
print("\n→ 死亡率較低（~16%），OR 和 RR 的差距比感染預測（43%）時小得多")

## 題目 2：多變項 Adjusted RR + Adjusted OR

建立預測死亡的多變項模型，同時用 Modified Poisson 和 Logistic Regression，
並排比較 adjusted RR 和 adjusted OR。

In [ ]:
# --- 共用公式 ---
formula_death = (
    "dead ~ age + comorbidity_chf + comorbidity_copd + "
    "immunosuppressed + severity_score"
)

# --- Modified Poisson → Adjusted RR ---
poisson_multi = smf.glm(
    formula_death, data=cases,
    family=sm.families.Poisson()
).fit(cov_type="HC0", disp=0)

# --- Logistic → Adjusted OR ---
logit_multi = smf.logit(formula_death, data=cases).fit(disp=0, method="lbfgs")

# --- 並排比較表格 ---
compare_rows = []
for var in poisson_multi.params.index:
    if var == "Intercept":
        continue
    # Adjusted RR
    rr = np.exp(poisson_multi.params[var])
    rr_ci = np.exp(poisson_multi.conf_int().loc[var])
    # Adjusted OR
    or_val = np.exp(logit_multi.params[var])
    or_ci = np.exp(logit_multi.conf_int().loc[var])
    # OR 比 RR 高估多少
    pct_diff = (or_val - rr) / rr * 100

    compare_rows.append({
        "variable": var,
        "adj_RR": round(rr, 3),
        "RR 95% CI": f"{rr_ci[0]:.3f}\u2013{rr_ci[1]:.3f}",
        "adj_OR": round(or_val, 3),
        "OR 95% CI": f"{or_ci[0]:.3f}\u2013{or_ci[1]:.3f}",
        "OR高估%": f"{pct_diff:+.1f}%",
    })

compare_df = pd.DataFrame(compare_rows)
print("=== 死亡預測：Adjusted RR vs Adjusted OR ===")
print(compare_df.to_string(index=False))

# --- Crude vs Adjusted 比較 ---
print("\n=== Crude → Adjusted 變化（用 RR） ===")
for var in factors_death:
    c_row = crude_df[crude_df["variable"] == var].iloc[0]
    a_row = compare_df[compare_df["variable"] == var]
    if len(a_row) == 0:
        continue
    a_row = a_row.iloc[0]
    change = (a_row["adj_RR"] - c_row["crude_RR"]) / c_row["crude_RR"] * 100
    print(f"  {var:25s}  crude_RR={c_row['crude_RR']:.3f}  "
          f"adj_RR={a_row['adj_RR']:.3f}  ({change:+.1f}%)")
print("\n→ 變化最大的變項 = 受其他因子干擾最多的因子")

## 題目 3（挑戰題）：模型比較 + Forest Plot

1. 建立兩個 Modified Poisson 模型（精簡 vs 完整）
2. 比較 AIC
3. 用較好的模型畫 Adjusted RR 森林圖

In [ ]:
# --- 模型 A（精簡）：3 個預測因子 ---
model_a = smf.glm(
    "dead ~ age + immunosuppressed + severity_score",
    data=cases, family=sm.families.Poisson()
).fit(cov_type="HC0", disp=0)

# --- 模型 B（完整）：5 個預測因子 ---
model_b = smf.glm(
    "dead ~ age + comorbidity_chf + comorbidity_copd + "
    "immunosuppressed + severity_score",
    data=cases, family=sm.families.Poisson()
).fit(cov_type="HC0", disp=0)

# --- AIC 比較 ---
print("=== 模型比較（Modified Poisson）===")
print(f"  模型 A（3 變項）AIC = {model_a.aic:.1f}")
print(f"  模型 B（5 變項）AIC = {model_b.aic:.1f}")

best = model_a if model_a.aic < model_b.aic else model_b
best_name = "A" if model_a.aic < model_b.aic else "B"
print(f"  → 模型 {best_name} 較佳（AIC 較小 = 解釋力與簡約的最佳平衡）")

In [ ]:
# --- Forest Plot：Adjusted RR（用較好的模型）---
forest_data = []
for var in best.params.index:
    if var == "Intercept":
        continue
    rr = np.exp(best.params[var])
    ci = np.exp(best.conf_int().loc[var])
    forest_data.append({
        "variable": var,
        "RR": rr,
        "ci_lo": ci[0],
        "ci_hi": ci[1],
    })

fdf = pd.DataFrame(forest_data)

fig, ax = plt.subplots(figsize=(8, 4))
y_pos = range(len(fdf))

# 點估計 + 信賴區間
ax.errorbar(
    fdf["RR"], y_pos,
    xerr=[fdf["RR"] - fdf["ci_lo"], fdf["ci_hi"] - fdf["RR"]],
    fmt="o", color="#D97757", capsize=4, markersize=8,
    ecolor="#6A9BCC", elinewidth=2,
)

# RR = 1 參考線（無效應）
ax.axvline(x=1, color="gray", linestyle="--", alpha=0.5, label="RR = 1")

ax.set_yticks(list(y_pos))
ax.set_yticklabels(fdf["variable"])
ax.set_xlabel("Adjusted Risk Ratio (RR)")
ax.set_title(f"死亡預測模型 {best_name} — Adjusted RR 森林圖")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

# --- 解讀 ---
print("\n=== 獨立預測因子（RR > 1 且 CI 不包含 1）===")
for _, row in fdf.iterrows():
    sig = "✓ 顯著" if row["ci_lo"] > 1 else "  不顯著"
    print(f"  {row['variable']:25s}  RR={row['RR']:.3f}  "
          f"({row['ci_lo']:.3f}\u2013{row['ci_hi']:.3f})  {sig}")

### 解讀

- **severity_score**：臨床嚴重度是死亡最強的預測因子（RR 最大），這符合直覺
- **immunosuppressed**：控制嚴重度後，免疫抑制可能仍為獨立危險因子
- **age**：年齡每增加一歲的 RR 看起來接近 1，但累積效應大（例如 80 歲 vs 70 歲差 10 歲）
- **RR vs OR**：死亡率 ~16%，OR 和 RR 差距比感染預測（侵襲率 43%）時小，驗證了「盛行率越低，OR 越接近 RR」的原則
- **模型選擇**：AIC 較小的模型不一定每個變項都顯著，但整體平衡較好
- **限制**：死亡人數只有 ~19 人，模型自由度有限，不宜放太多變項

## 題目 4 解答

In [ ]:
# --- 資料：結核病接觸者疫調（合成資料，與練習題相同）---
rng = np.random.default_rng(406)
n = 600

age = rng.normal(45, 15, n).clip(5, 90)
diabetes = (rng.uniform(0, 1, n) < (0.05 + age / 300)).astype(int)
close_contact = rng.binomial(1, 0.4, n)
underweight = rng.binomial(1, 0.15, n)
smoking = rng.binomial(1, 0.25, n)

logit_p = (
    -4.3
    + 0.03 * age
    + 0.8 * diabetes
    + 1.3 * close_contact
    + 0.9 * underweight
    + 0.5 * smoking
)
p_tb = 1 / (1 + np.exp(-logit_p))
active_tb = rng.binomial(1, p_tb)

tb = pd.DataFrame({
    "age": age.round(1),
    "diabetes": diabetes,
    "close_contact": close_contact,
    "underweight": underweight,
    "smoking": smoking,
    "active_tb": active_tb,
})

prevalence = tb["active_tb"].mean()
print(f"樣本數：{len(tb)}，活動性結核：{tb['active_tb'].sum()} 人")
print(f"盛行率：{prevalence:.1%}\n")

# --- Crude OR：close_contact ---
crude_fit = smf.logit("active_tb ~ close_contact", data=tb).fit(disp=0)
crude_or = np.exp(crude_fit.params["close_contact"])
crude_ci = np.exp(crude_fit.conf_int().loc["close_contact"])
print(f"Crude OR (close_contact) = {crude_or:.2f}  "
      f"(95% CI: {crude_ci[0]:.2f}\u2013{crude_ci[1]:.2f})")

# --- 多變項模型：Adjusted OR ---
multi_fit = smf.logit(
    "active_tb ~ age + diabetes + close_contact + underweight + smoking",
    data=tb,
).fit(disp=0)

rows = []
for var in multi_fit.params.index:
    if var == "Intercept":
        continue
    or_val = np.exp(multi_fit.params[var])
    ci = np.exp(multi_fit.conf_int().loc[var])
    rows.append({
        "variable": var,
        "adj_OR": round(or_val, 3),
        "95% CI": f"{ci[0]:.3f}\u2013{ci[1]:.3f}",
        "significant": "是" if (ci[0] > 1 or ci[1] < 1) else "否",
    })
tb_result = pd.DataFrame(rows)
print("\n=== 多變項模型：Adjusted OR ===")
print(tb_result.to_string(index=False))

adj_or_contact = np.exp(multi_fit.params["close_contact"])
change_pct = (adj_or_contact - crude_or) / crude_or * 100
print(f"\ncrude OR = {crude_or:.2f} -> adjusted OR = {adj_or_contact:.2f}  ({change_pct:+.1f}%)")
if abs(change_pct) > 10:
    print("→ close_contact 的效應在校正後有明顯改變，顯示存在干擾因子（如年齡、糖尿病）")
else:
    print("→ close_contact 的效應校正前後變化不大，干擾程度較小")

print("\n=== 校正後達統計顯著的危險因子（95% CI 不含 1）===")
sig = tb_result[tb_result["significant"] == "是"]
print(sig.to_string(index=False))
print(f"\n→ adjusted OR 最大的危險因子：{tb_result.loc[tb_result['adj_OR'].idxmax(), 'variable']}")

## 題目 5 解答

In [ ]:
# --- 資料：COVID-19 社區篩檢確診病例（合成資料，與練習題相同）---
rng = np.random.default_rng(507)
n = 600

age = rng.normal(50, 18, n).clip(18, 95)
obesity = rng.binomial(1, 0.30, n)
unvaccinated = rng.binomial(1, 0.35, n)
chronic_lung = rng.binomial(1, 0.12, n)

logit_p = (
    -5.3
    + 0.05 * age
    + 0.8 * obesity
    + 1.1 * unvaccinated
    + 0.9 * chronic_lung
)
p_severe = 1 / (1 + np.exp(-logit_p))
severe = rng.binomial(1, p_severe)

covid = pd.DataFrame({
    "age": age.round(1),
    "obesity": obesity,
    "unvaccinated": unvaccinated,
    "chronic_lung": chronic_lung,
    "severe": severe,
})

severe_rate = covid["severe"].mean()
print(f"樣本數：{len(covid)}，重症人數：{covid['severe'].sum()}")
print(f"重症比例：{severe_rate:.1%}\n")

# --- Crude OR：unvaccinated ---
crude_fit = smf.logit("severe ~ unvaccinated", data=covid).fit(disp=0)
crude_or = np.exp(crude_fit.params["unvaccinated"])
crude_ci = np.exp(crude_fit.conf_int().loc["unvaccinated"])
print(f"Crude OR (unvaccinated) = {crude_or:.2f}  "
      f"(95% CI: {crude_ci[0]:.2f}\u2013{crude_ci[1]:.2f})")

# --- 多變項模型：Adjusted OR ---
multi_fit = smf.logit(
    "severe ~ age + obesity + unvaccinated + chronic_lung", data=covid
).fit(disp=0)

rows = []
for var in multi_fit.params.index:
    if var == "Intercept":
        continue
    or_val = np.exp(multi_fit.params[var])
    ci = np.exp(multi_fit.conf_int().loc[var])
    rows.append({
        "variable": var,
        "adj_OR": round(or_val, 3),
        "95% CI": f"{ci[0]:.3f}\u2013{ci[1]:.3f}",
    })
covid_result = pd.DataFrame(rows)
print("\n=== 多變項模型：Adjusted OR ===")
print(covid_result.to_string(index=False))

adj_or_unvax = np.exp(multi_fit.params["unvaccinated"])
adj_ci_unvax = np.exp(multi_fit.conf_int().loc["unvaccinated"])
print(f"\ncrude OR = {crude_or:.2f} -> adjusted OR = {adj_or_unvax:.2f}  "
      f"(95% CI: {adj_ci_unvax[0]:.2f}\u2013{adj_ci_unvax[1]:.2f})")

print("\n=== 解讀 ===")
if adj_ci_unvax[0] > 1:
    print("→ unvaccinated 的 adjusted OR 顯著大於 1（95% CI 不含 1），")
    print("  表示校正年齡、肥胖、慢性肺病後，未接種疫苗者發生重症的勝算")
    print(f"  仍是接種者的 {adj_or_unvax:.1f} 倍，支持「接種疫苗能降低重症風險」的結論")
else:
    print("→ unvaccinated 的 95% CI 包含 1，本次樣本未能證實顯著關聯")

## 題目 6 解答

In [ ]:
# --- 資料：登革熱確診病例（合成資料，與練習題相同）---
rng = np.random.default_rng(608)
n = 550

secondary_infection = rng.binomial(1, 0.30, n)
age = rng.normal(35, 20, n).clip(1, 85)
diabetes = rng.binomial(1, 0.15, n)
hypertension = rng.binomial(1, 0.20, n)

logit_p = (
    -3.8
    + 1.5 * secondary_infection
    + 0.03 * age
    + 0.7 * diabetes
    + 0.5 * hypertension
)
p_severe = 1 / (1 + np.exp(-logit_p))
severe_dengue = rng.binomial(1, p_severe)

dengue = pd.DataFrame({
    "secondary_infection": secondary_infection,
    "age": age.round(1),
    "diabetes": diabetes,
    "hypertension": hypertension,
    "severe_dengue": severe_dengue,
})

severe_rate = dengue["severe_dengue"].mean()
print(f"樣本數：{len(dengue)}，重症登革熱：{dengue['severe_dengue'].sum()} 人")
print(f"重症比例：{severe_rate:.1%}\n")

# --- Crude OR：secondary_infection ---
crude_fit = smf.logit("severe_dengue ~ secondary_infection", data=dengue).fit(disp=0)
crude_or = np.exp(crude_fit.params["secondary_infection"])
crude_ci = np.exp(crude_fit.conf_int().loc["secondary_infection"])
print(f"Crude OR (secondary_infection) = {crude_or:.2f}  "
      f"(95% CI: {crude_ci[0]:.2f}\u2013{crude_ci[1]:.2f})")

# --- 多變項模型：Adjusted OR ---
multi_fit = smf.logit(
    "severe_dengue ~ secondary_infection + age + diabetes + hypertension", data=dengue
).fit(disp=0)

rows = []
for var in multi_fit.params.index:
    if var == "Intercept":
        continue
    or_val = np.exp(multi_fit.params[var])
    ci = np.exp(multi_fit.conf_int().loc[var])
    rows.append({
        "variable": var,
        "adj_OR": round(or_val, 3),
        "95% CI": f"{ci[0]:.3f}\u2013{ci[1]:.3f}",
    })
dengue_result = pd.DataFrame(rows)
print("\n=== 多變項模型：Adjusted OR ===")
print(dengue_result.to_string(index=False))

adj_or_2nd = np.exp(multi_fit.params["secondary_infection"])
adj_ci_2nd = np.exp(multi_fit.conf_int().loc["secondary_infection"])
change_pct = (adj_or_2nd - crude_or) / crude_or * 100
print(f"\ncrude OR = {crude_or:.2f} -> adjusted OR = {adj_or_2nd:.2f}  "
      f"(95% CI: {adj_ci_2nd[0]:.2f}\u2013{adj_ci_2nd[1]:.2f})  ({change_pct:+.1f}%)")

print("\n=== 解讀 ===")
print(f"→ 校正年齡、糖尿病、高血壓後，二次感染者發生重症登革熱的勝算")
print(f"  是初次感染者的 {adj_or_2nd:.1f} 倍")
print("→ 此結果與 ADE（抗體依賴增強作用：對第一種血清型的非中和抗體")
print("  反而促進第二種血清型病毒進入細胞複製）之病理機轉一致"
      if adj_or_2nd > 1 else "→ 本次樣本未能觀察到二次感染的加成效應")

## 題目 7 解答

In [ ]:
# --- 資料：流感確診病例（合成資料，與練習題相同）---
rng = np.random.default_rng(709)
n = 700

age = rng.normal(40, 20, n).clip(0, 95)
chronic_disease = rng.binomial(1, 0.22, n)
p_vaccinated = 0.20 + 0.65 * chronic_disease
vaccinated = rng.binomial(1, p_vaccinated)
late_treatment = rng.binomial(1, 0.40, n)

logit_p = (
    -4.1
    + 0.03 * age
    + 1.8 * chronic_disease
    - 0.65 * vaccinated
    + 0.9 * late_treatment
)
p_hosp = 1 / (1 + np.exp(-logit_p))
hospitalized = rng.binomial(1, p_hosp)

flu = pd.DataFrame({
    "age": age.round(1),
    "chronic_disease": chronic_disease,
    "vaccinated": vaccinated,
    "late_treatment": late_treatment,
    "hospitalized": hospitalized,
})

hosp_rate = flu["hospitalized"].mean()
print(f"樣本數：{len(flu)}，住院人數：{flu['hospitalized'].sum()}")
print(f"住院比例：{hosp_rate:.1%}\n")

# --- Crude OR：vaccinated ---
crude_fit = smf.logit("hospitalized ~ vaccinated", data=flu).fit(disp=0)
crude_or = np.exp(crude_fit.params["vaccinated"])
crude_ci = np.exp(crude_fit.conf_int().loc["vaccinated"])
print(f"Crude OR (vaccinated) = {crude_or:.2f}  "
      f"(95% CI: {crude_ci[0]:.2f}\u2013{crude_ci[1]:.2f})")

# --- 多變項模型：Adjusted OR ---
multi_fit = smf.logit(
    "hospitalized ~ age + chronic_disease + vaccinated + late_treatment", data=flu
).fit(disp=0)

rows = []
for var in multi_fit.params.index:
    if var == "Intercept":
        continue
    or_val = np.exp(multi_fit.params[var])
    ci = np.exp(multi_fit.conf_int().loc[var])
    rows.append({
        "variable": var,
        "adj_OR": round(or_val, 3),
        "95% CI": f"{ci[0]:.3f}\u2013{ci[1]:.3f}",
    })
flu_result = pd.DataFrame(rows)
print("\n=== 多變項模型：Adjusted OR ===")
print(flu_result.to_string(index=False))

adj_or_vax = np.exp(multi_fit.params["vaccinated"])
adj_ci_vax = np.exp(multi_fit.conf_int().loc["vaccinated"])
print(f"\ncrude OR (vaccinated) = {crude_or:.2f}  ->  adjusted OR = {adj_or_vax:.2f}  "
      f"(95% CI: {adj_ci_vax[0]:.2f}\u2013{adj_ci_vax[1]:.2f})")

print("\n=== 解讀 ===")
if crude_or >= 1 and adj_or_vax < 1:
    print("→ crude OR 顯示接種者住院風險較高（或接近 1），")
    print("  但校正慢性病史後，adjusted OR 反轉為保護效果（OR < 1）")
else:
    print("→ crude OR 與 adjusted OR 方向一致，但仍可比較兩者差距")
print("→ 此現象稱為「適應症干擾」（confounding by indication）：")
print("  慢性病患者因本身風險較高而被優先建議接種疫苗，")
print("  使 vaccinated 與 chronic_disease 正相關，若未校正 chronic_disease，")
print("  crude OR 會低估（甚至反轉）疫苗真正的保護效果")

## 題目 8 解答

In [ ]:
# --- 資料：麻疹群突發病例（合成資料，與練習題相同）---
rng = np.random.default_rng(810)
n = 650

age = rng.uniform(0.5, 15, n)
malnutrition = rng.binomial(1, 0.25, n)
vitamin_a_deficiency = rng.binomial(1, 0.20, n)

logit_p = (
    -2.8
    + 0.6 * malnutrition
    + 0.6 * vitamin_a_deficiency
    + 1.6 * malnutrition * vitamin_a_deficiency
    - 0.05 * age
)
p_severe = 1 / (1 + np.exp(-logit_p))
severe_complication = rng.binomial(1, p_severe)

measles = pd.DataFrame({
    "age": age.round(2),
    "malnutrition": malnutrition,
    "vitamin_a_deficiency": vitamin_a_deficiency,
    "severe_complication": severe_complication,
})


# --- 分組比較（觀察加乘效應）---
def group_label(row):
    if row["malnutrition"] and row["vitamin_a_deficiency"]:
        return "兩者皆有"
    if row["malnutrition"]:
        return "僅營養不良"
    if row["vitamin_a_deficiency"]:
        return "僅維生素A缺乏"
    return "皆無"


measles["group"] = measles.apply(group_label, axis=1)
group_summary = measles.groupby("group")["severe_complication"].agg(["mean", "count"])
group_summary = group_summary.rename(columns={"mean": "重症比例", "count": "人數"})
print("=== 各組重症併發症比例 ===")
print(group_summary.to_string())
print("\n→ 若「兩者皆有」的比例遠高於「僅營養不良」與「僅維生素A缺乏」單獨效果的加總，")
print("  代表兩者在風險上可能有加乘（超相乘）作用\n")

# --- 主效應模型（無交互作用）---
main_fit = smf.logit(
    "severe_complication ~ malnutrition + vitamin_a_deficiency + age", data=measles
).fit(disp=0)

print("=== 主效應模型 Adjusted OR ===")
for var in ["malnutrition", "vitamin_a_deficiency", "age"]:
    or_val = np.exp(main_fit.params[var])
    ci = np.exp(main_fit.conf_int().loc[var])
    print(f"  {var:22s}  OR={or_val:.3f}  (95% CI: {ci[0]:.3f}\u2013{ci[1]:.3f})")
print(f"  AIC = {main_fit.aic:.1f}\n")

# --- 交互作用模型 ---
inter_fit = smf.logit(
    "severe_complication ~ malnutrition * vitamin_a_deficiency + age", data=measles
).fit(disp=0)

print("=== 交互作用模型 Adjusted OR ===")
for var in inter_fit.params.index:
    if var == "Intercept":
        continue
    or_val = np.exp(inter_fit.params[var])
    ci = np.exp(inter_fit.conf_int().loc[var])
    print(f"  {var:40s}  OR={or_val:.3f}  (95% CI: {ci[0]:.3f}\u2013{ci[1]:.3f})")
print(f"  AIC = {inter_fit.aic:.1f}")

interaction_term = "malnutrition:vitamin_a_deficiency"
inter_or = np.exp(inter_fit.params[interaction_term])
inter_ci = np.exp(inter_fit.conf_int().loc[interaction_term])

print("\n=== 模型比較 ===")
print(f"主效應模型 AIC = {main_fit.aic:.1f}")
print(f"交互作用模型 AIC = {inter_fit.aic:.1f}")
if inter_fit.aic < main_fit.aic:
    print("→ 交互作用模型 AIC 較小，加入交互作用項改善了模型適配度")
else:
    print("→ 交互作用模型未明顯改善 AIC，需視樣本數與信賴區間寬度綜合判斷")

print("\n=== 解讀 ===")
print(f"交互作用項 (malnutrition:vitamin_a_deficiency) OR = {inter_or:.2f}  "
      f"(95% CI: {inter_ci[0]:.2f}\u2013{inter_ci[1]:.2f})")
print("→ 此 OR 代表「兩因子同時存在」相對於「兩個主效應各自獨立相乘」之後，")
print("  額外增加的相乘倍數（超相乘交互作用，supra-multiplicative interaction）")
if inter_or > 1:
    print("→ OR > 1，顯示營養不良與維生素A缺乏同時存在時，重症風險的加乘幅度")
    print("  超過兩者個別效應的乘積，兩者在此模型中呈現加乘（協同）作用")
else:
    print("→ OR ≤ 1，本次樣本未能觀察到加乘作用")
print("→ 公衛意義：篩檢麻疹病例時應優先辨識「營養不良合併維生素A缺乏」的兒童，")
print("  及早補充維生素A並密切追蹤，以降低重症併發症風險")